# GLM-OCR: base vs LoRA on CoSyn chemical molecules

Runs the whole experiment on a Colab GPU: baseline eval, LoRA training, fine-tuned eval, summary.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
!git clone https://github.com/MahnoorAhmed-Dev/glm-ocr-lora-cosyn-chem.git
%cd glm-ocr-lora-cosyn-chem
!pip install -q -e LLaMA-Factory rdkit
# transformers needs huggingface-hub 1.x; do not upgrade it to 2.x
!pip install -q "huggingface_hub>=1.5,<2.0"
!hf download zai-org/GLM-OCR --local-dir GLM-OCR

In [ ]:
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 2. Baseline: untouched GLM-OCR on the held-out set (30 molecules)

In [ ]:
!python scripts/evaluate.py     --data LLaMA-Factory/data/cosyn_chem_eval.json --data-root LLaMA-Factory/data     --model GLM-OCR --out results/base.json

## 3. LoRA fine-tune (200 training molecules, 3 epochs)

In [ ]:
%cd LLaMA-Factory
!llamafactory-cli train data/glm_ocr_lora_colab.yaml
%cd ..

## 4. Fine-tuned: base + LoRA adapter on the same held-out set

In [ ]:
!python scripts/evaluate.py     --data LLaMA-Factory/data/cosyn_chem_eval.json --data-root LLaMA-Factory/data     --model GLM-OCR --adapter LLaMA-Factory/saves/glm-ocr/lora/colab --out results/lora.json

## 5. Summary and download

In [ ]:
!python scripts/make_summary.py
from IPython.display import Markdown
Markdown(open("results/summary.md").read())

In [ ]:
# Bundle only the small files worth committing: results + the LoRA adapter (+ loss plot).
!mkdir -p bundle/adapter
!cp results/*.json results/summary.md bundle/
!cp LLaMA-Factory/saves/glm-ocr/lora/colab/{adapter_config.json,adapter_model.safetensors,training_loss.png,trainer_log.jsonl,all_results.json} bundle/adapter/ 2>/dev/null
!cd bundle && zip -rq ../results_bundle.zip . && ls -la ../results_bundle.zip
from google.colab import files
files.download("results_bundle.zip")